# 02 — Cleaning (Scrub)

Converts the raw scrape (`data/raw/grand_tours_raw.csv`, 333 rows) into a tidy, typed
`editions` table.

**Principle**: no row is dropped. Rows that represent something other than a completed
edition are *classified*, not deleted, so each analysis can choose its own filter.

The parsing functions live in `src/cleaning.py` and are covered by 25 unit tests in
`tests/test_cleaning.py` — every edge case documented in notebook 01 has a corresponding test.

In [64]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
import numpy as np

from src.cleaning import (
    clean_missing, parse_distance, parse_time_hours,
    parse_points, parse_rider_name,
)

raw = pd.read_csv("../data/raw/grand_tours_raw.csv")
print(raw.shape)
raw.head()

(333, 10)


,race,year,country,cyclist,team,distance,time_points,margin,stage_wins,title_reassigned
0,tdf,1903,France,Maurice Garin,La Française,"2,428km (1,509mi)",94h 33′ 14″,+ 2h 59′ 21″,3,False
1,tdf,1904,France,Henri Cornet[b],Conte,"2,428km (1,509mi)",96h 05′ 55″,+ 2h 16′ 14″,1,False
2,tdf,1905,France,Louis Trousselier,Peugeot–Wolber,"2,994km (1,860mi)",35,26,5,False
3,tdf,1906,France,René Pottier,Peugeot–Wolber,"4,637km (2,881mi)",31,8,5,False
4,tdf,1907,France,Lucien Petit-Breton,Peugeot–Wolber,"4,488km (2,789mi)",47,19,2,False


## Step 1 — the em-dash problem

Before anything else: pandas reports almost no missing values in the raw data, because every
gap is the string `'—'` rather than an empty cell. Until those are converted, `isna()` is
actively misleading.

In [65]:
print("Missing values as pandas sees them (raw):")
print(raw.isna().sum(), "\n")

print("Cells actually containing an em-dash:")
print((raw == "—").sum())

Missing values as pandas sees them (raw):
race                 0
year                 0
country              0
cyclist              0
team                 0
distance             0
time_points          0
margin               0
stage_wins          22
title_reassigned     0
dtype: int64 

Cells actually containing an em-dash:
race                 0
year                 0
country             38
cyclist             24
team                47
distance            31
time_points         38
margin              38
stage_wins          16
title_reassigned     0
dtype: int64


## Step 2 — parse the name field into three variables

The `cyclist` column encodes three separate facts: who won, whether the title was awarded
retroactively (`#` / `†`), and whether the result was annulled (`No winner`). Tidy data
requires one variable per column, so it is split into three.

In [66]:
name_parts = raw["cyclist"].apply(parse_rider_name).apply(pd.Series)
print(name_parts["multi_classification"].sum(), "riders also won another classification")
print(name_parts["no_winner"].sum(), "annulled results")

name_parts[name_parts["multi_classification"]].join(
    raw[["race", "year"]]
).head(15)

68 riders also won another classification
7 annulled results


,rider_name,multi_classification,no_winner,race,year
35,Gino Bartali,True,False,tdf,1938
36,Sylvère Maes,True,False,tdf,1939
45,Gino Bartali,True,False,tdf,1948
46,Fausto Coppi,True,False,tdf,1949
49,Fausto Coppi,True,False,tdf,1952
56,Federico Bahamontes,True,False,tdf,1959
67,Eddy Merckx,True,False,tdf,1970
68,Eddy Merckx,True,False,tdf,1971
69,Eddy Merckx,True,False,tdf,1972
76,Bernard Hinault,True,False,tdf,1979


## Step 3 — build the tidy editions table

Note that `time_points` is passed to *both* a time parser and a points parser. They are
complementary: early editions were scored on points and have no elapsed time, later ones the
reverse. Exactly one of the two returns a value for any given row.

In [67]:
editions = pd.DataFrame({
    "race": raw["race"],
    "year": raw["year"].astype(int),
    "rider_name": name_parts["rider_name"],
    "country": raw["country"].apply(clean_missing),
    "team": raw["team"].apply(clean_missing),
    "distance_km": raw["distance"].apply(parse_distance),
    "winning_time_hours": raw["time_points"].apply(parse_time_hours),
    "winning_points": raw["time_points"].apply(parse_points),
    "multi_classification": name_parts["multi_classification"],
    "title_reassigned": raw["title_reassigned"],
    "no_winner": name_parts["no_winner"],
})

print(editions.shape)
editions.head()

(333, 11)


,race,year,rider_name,country,team,distance_km,winning_time_hours,winning_points,multi_classification,title_reassigned,no_winner
0,tdf,1903,Maurice Garin,France,La Française,2428.0,94.553889,NaN,False,False,False
1,tdf,1904,Henri Cornet,France,Conte,2428.0,96.098611,NaN,False,False,False
2,tdf,1905,Louis Trousselier,France,Peugeot–Wolber,2994.0,NaN,35.0,False,False,False
3,tdf,1906,René Pottier,France,Peugeot–Wolber,4637.0,NaN,31.0,False,False,False
4,tdf,1907,Lucien Petit-Breton,France,Peugeot–Wolber,4488.0,NaN,47.0,False,False,False


## Step 4 — classify each row

Three distinct kinds of row, which must not be conflated:

| Status | Meaning | Example |
|---|---|---|
| `held` | A normal edition with a winner | 1903 TdF |
| `disputed` | Race took place; result annulled | TdF 1999–2005 |
| `no_race` | No edition took place | War years |

The distinction matters. The 1999–2005 Tours had routes, distances and finishers — they
belong in any analysis of race distance, but not in any count of winners. A war year belongs
in neither.

In [68]:
def classify(row):
    if row["no_winner"]:
        return "disputed"
    if pd.isna(row["rider_name"]):
        return "no_race"
    return "held"

editions["status"] = editions.apply(classify, axis=1)
print(editions.groupby(["race", "status"]).size().unstack(fill_value=0))

status  disputed  held  no_race
race                           
giro           0   108       10
tdf            7   106       11
vuelta         0    80       11


In [69]:
# 1912 Giro: race was held, but scored as a team classification only,
# so there is no individual winner. It is not a war-year gap.
mask_1912 = (editions["race"] == "giro") & (editions["year"] == 1912)
print(editions.loc[mask_1912].T)

editions.loc[mask_1912, "status"] = "team_only"
print("\nStatus counts after correction:")
print(editions["status"].value_counts())

                               127
race                          giro
year                          1912
rider_name                     NaN
country                      Italy
team                  Atala–Dunlop
distance_km                 2443.0
winning_time_hours             NaN
winning_points                33.0
multi_classification         False
title_reassigned             False
no_winner                    False
status                     no_race

Status counts after correction:
status
held         294
no_race       31
disputed       7
team_only      1
Name: count, dtype: int64


## Step 5 — derived metric: average speed

Winning *times* cannot be compared across eras, because they measure route length as much as
rider performance. The 1926 Tour took 238 hours over 5,745 km; a modern Tour takes ~80 hours
over ~3,300 km. The modern rider is not four times faster — the race is simply shorter.

Average speed (distance ÷ time) removes route length from the comparison and leaves something
closer to actual performance. It is only defined for editions with both a distance and a
recorded time, so points-scored years are excluded automatically.

In [70]:
editions["avg_speed_kmh"] = editions["distance_km"] / editions["winning_time_hours"]

print("Editions with a computable speed:", editions["avg_speed_kmh"].notna().sum())
print("\nSpeed range:")
print(editions["avg_speed_kmh"].describe().round(2))

print("\nFastest 5 editions:")
print(editions.nlargest(5, "avg_speed_kmh")[
    ["race", "year", "rider_name", "distance_km", "winning_time_hours", "avg_speed_kmh"]
].round(2).to_string(index=False))

print("\nSlowest 5 editions:")
print(editions.nsmallest(5, "avg_speed_kmh")[
    ["race", "year", "rider_name", "distance_km", "winning_time_hours", "avg_speed_kmh"]
].round(2).to_string(index=False))

Editions with a computable speed: 282

Speed range:
count    282.00
mean      35.60
std        4.72
min       23.37
25%       33.44
50%       36.52
75%       39.22
max       44.45
Name: avg_speed_kmh, dtype: float64

Fastest 5 editions:
  race  year       rider_name  distance_km  winning_time_hours  avg_speed_kmh
vuelta  2025 Jonas Vingegaard       3304.3               74.34          44.45
   tdf  2026    Tadej Pogačar       3245.0               73.94          43.89
   tdf  2025    Tadej Pogačar       3320.0               76.01          43.68
vuelta  2003    Roberto Heras       2958.3               69.53          42.55
vuelta  2001     Ángel Casero       3012.2               70.82          42.53

Slowest 5 editions:
race  year         rider_name  distance_km  winning_time_hours  avg_speed_kmh
giro  1914  Alfonso Calzolari       3162.0              135.30          23.37
 tdf  1924 Ottavio Bottecchia       5425.0              226.31          23.97
 tdf  1919      Firmin Lambot       5560

## Step 6 — build the riders table

The second tidy table: one row per rider, with win counts per race. This is what makes
cross-race questions answerable — "who has won more than one different Grand Tour" requires
a rider-level view that the original project's four separate files could not produce.

Only `held` editions count toward wins. Disputed years have no winner to credit, and the
1912 Giro had no individual classification.

In [71]:
held = editions[editions["status"] == "held"]

wins = (held.groupby(["rider_name", "race"])
             .size()
             .unstack(fill_value=0)
             .rename(columns={"tdf": "wins_tdf", "giro": "wins_giro", "vuelta": "wins_vuelta"}))

wins["wins_total"] = wins.sum(axis=1)
wins["races_won"] = (wins[["wins_tdf", "wins_giro", "wins_vuelta"]] > 0).sum(axis=1)

# attach each rider's country (taken from their first recorded win)
countries = held.groupby("rider_name")["country"].first()
riders = wins.join(countries).reset_index()
    
print(riders.shape)
print("\nMost successful riders:")
print(riders.nlargest(10, "wins_total")[
    ["rider_name", "country", "wins_tdf", "wins_giro", "wins_vuelta", "wins_total", "races_won"]
].to_string(index=False))

print("\nRiders who won all three Grand Tours:")
print(riders[riders["races_won"] == 3][
    ["rider_name", "country", "wins_tdf", "wins_giro", "wins_vuelta"]
].to_string(index=False))

(164, 7)

Most successful riders:
      rider_name       country  wins_tdf  wins_giro  wins_vuelta  wins_total  races_won
 Bernard Hinault        France         5          3            2          10          3
     Eddy Merckx       Belgium         4          4            1           9          3
Jacques Anquetil        France         5          2            1           8          3
Alberto Contador         Spain         2          2            3           7          3
    Chris Froome Great Britain         4          1            2           7          3
    Fausto Coppi         Italy         2          5            0           7          2
   Alfredo Binda         Italy         0          5            0           5          1
  Felice Gimondi         Italy         1          3            1           5          3
    Gino Bartali         Italy         2          3            0           5          2
 Miguel Induráin         Spain         5          0            0           5          

In [72]:
riders = riders[["rider_name", "country", "wins_tdf", "wins_giro",
                 "wins_vuelta", "wins_total", "races_won"]]

checks = {
    "no rows lost": len(editions) == 333,
    "status accounts for all rows": editions["status"].value_counts().sum() == 333,
    "8 riders won all three": (riders["races_won"] == 3).sum() == 8,
    "speeds are plausible (20-50 km/h)":
        editions["avg_speed_kmh"].dropna().between(20, 50).all(),
    "no rider name contains markup":
        not editions["rider_name"].dropna().str.contains(r"[#†*~&\[\]]").any(),
    "held editions all have a rider":
        editions.loc[editions["status"] == "held", "rider_name"].notna().all(),
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")

PASS  no rows lost
PASS  status accounts for all rows
PASS  8 riders won all three
PASS  speeds are plausible (20-50 km/h)
PASS  no rider name contains markup
PASS  held editions all have a rider


In [73]:
editions.to_csv("../data/processed/editions.csv", index=False)
riders.to_csv("../data/processed/riders.csv", index=False)

print("editions:", editions.shape)
print("riders  :", riders.shape)

editions: (333, 13)
riders  : (164, 7)


In [74]:
print(raw["title_reassigned"].value_counts())
print(raw["title_reassigned"].dtype)

title_reassigned
False    331
True       2
Name: count, dtype: int64
bool


In [75]:
print("Reassigned titles found:", editions["title_reassigned"].sum())
print(editions[editions["title_reassigned"]][
    ["race", "year", "rider_name"]].to_string(index=False))

Reassigned titles found: 2
  race  year       rider_name
vuelta  1982 Marino Lejarreta
vuelta  2011     Chris Froome


### Debugging note: finding the real reassignment signal

Validating the rider table against a known fact — Wikipedia states that eight riders have won
all three Grand Tours — initially returned **six**. Chasing that gap exposed two separate bugs.

**1. A fifth marker symbol.** The Vuelta page uses `&` where the other two use `†`, `#` or `*`
(`Eddy Merckx&`). Each of these was discovered the same way: by noticing an implausible result
and tracing it back. Wikipedia's table notation is not documented in any schema; it has to be
reverse-engineered from the output.

**2. Reassigned titles are encoded in HTML, not notation.** The 2011 Vuelta produced the string
`'Juan José CoboChris Froome'` — two names glued together. Inspecting the markup showed why:

```html
<th><s><a href="...">Juan José Cobo</a></s> <a href="...">Chris Froome</a></th>
```

The stripped rider's name sits inside a `<s>` (strikethrough) tag. `get_text()` flattens the
element and concatenates both names with no separator.

The fix removes struck-through elements with `decompose()` *before* extracting text, and
records their presence as a `title_reassigned` flag. This finally produces the reassignment
data the earlier symbol-based attempt was reaching for — derived from the document's semantics
rather than from guessed notation.

**Result**: eight riders with all three Grand Tours, matching the source.

**The general lesson**: the discrepancy was only visible because the pipeline was checked against
an externally known fact. Without that check, the dataset would have looked entirely reasonable —
164 riders, plausible win counts, no errors raised — while silently missing two of cycling's
most decorated riders' Vuelta victories.